# Lecture 1: The necessary conditions, derived

**In this lecture.** We pose the cart-pole swing-up as an optimal control
problem and derive, by hand and with SymPy, everything an indirect method has
to integrate: the Hamiltonian, the costate equations, the optimal control law,
the boundary conditions, and then the two objects that make Newton's method
work, the state transition matrix and its jump at a switch. At the end we check
that `auto_optimal_control.derive` produces the same expressions and look at
the C it generates from them.

**Prerequisites.** Lagrange multipliers for constrained minimization, linear
ODEs, and enough mechanics to accept the cart-pole's equations of motion.

**After this lecture you can**

- write down the first-order necessary conditions for a fixed-time problem
  with fixed endpoints, and say where each one comes from;
- explain why bounded controls need the minimum principle instead of
  $H_u = 0$, and what the switching function is;
- explain what the state transition matrix is, why shooting needs it, and why
  it jumps where the control switches.

The argument follows the calculus-of-variations route of Kirk (2004,
chapter 5), as taught in MIT 16.323 (lecture notes 6 and 9) and Stanford
AA203 (lectures 4 and 5).

## 1. The problem

A cart of mass $m_1$ runs on a horizontal track. A pole of mass $m_2$ and
length $l$ hangs from it on a frictionless pivot. We push the cart with a
horizontal force $u$. The state is

$$
x = (q_1, q_2, \dot q_1, \dot q_2),
$$

the cart position, the pole angle ($q_2 = 0$ hanging down, $q_2 = \pi$
upright) and their rates. The task, from Kelly (2017): start at rest hanging
down, and arrive at rest upright, a distance $d$ down the track, at a fixed
time $T$, while spending as little effort as possible:

$$
\min_u \; J = \int_0^T u(t)^2 \, dt
$$

subject to

$$
\dot x = f(x, u), \qquad x(0) = 0, \qquad x(T) = (d, \pi, 0, 0).
$$

The parameter values are $m_1 = 1$ kg, $m_2 = 0.3$ kg, $l = 0.5$ m,
$g = 9.81$ m/s², $d = 1$ m and $T = 2$ s. The equations of motion are
Kelly's (6.1) and (6.2).

In [ ]:
import numpy as np
import sympy as sp

sp.init_printing()

t = sp.Symbol("t", real=True)
q1, q2, w1, w2, u = sp.symbols("q_1 q_2 qd_1 qd_2 u", real=True)
m1, m2, l, g, d, T = sp.symbols("m_1 m_2 l g d T", positive=True)

den = m1 + m2 * (1 - sp.cos(q2) ** 2)
q1dd = (
    l * m2 * sp.sin(q2) * w2**2 + u + m2 * g * sp.cos(q2) * sp.sin(q2)
) / den
q2dd = -(
    l * m2 * sp.cos(q2) * sp.sin(q2) * w2**2
    + u * sp.cos(q2)
    + (m1 + m2) * g * sp.sin(q2)
) / (l * den)

x = sp.Matrix([q1, q2, w1, w2])
f = sp.Matrix([w1, w2, q1dd, q2dd])
L = u**2
sp.Eq(sp.Derivative(x, t), f, evaluate=False)

Notice that $f$ is **affine in $u$**: $f(x, u) = f_0(x) + f_1(x)\,u$. That
will matter twice, once for the control law and once for the switching
function.

In [ ]:
f1 = f.diff(u)
f0 = sp.simplify(f - f1 * u)
f1.T

## 2. From Lagrange multipliers to the Hamiltonian

In finite dimensions, to minimize $\phi(y)$ subject to $c(y) = 0$ we adjoin
the constraint with a multiplier, $\phi + \lambda^\top c$, and ask that the
first variation vanish. Here the constraint is the differential equation, one
equation at each instant, so the multiplier becomes a function of time, the
**costate** $\lambda(t)$:

$$
J_a = \int_0^T \Big( L(x, u) + \lambda^\top \big(f(x, u) - \dot x\big) \Big)\, dt .
$$

Define the **Hamiltonian** $H(x, u, \lambda) = L(x, u) + \lambda^\top f(x, u)$.
Integrate the $-\lambda^\top \dot x$ term by parts. With $x(0)$ and $x(T)$
both fixed the boundary terms vanish, and the first variation is

$$
\delta J_a = \int_0^T \Big( (H_x + \dot\lambda^\top)\,\delta x
           + H_u\, \delta u + (H_\lambda - \dot x^\top)\,\delta\lambda \Big)\, dt .
$$

For $\delta J_a = 0$ for every admissible variation, each bracket must vanish.

> **First-order necessary conditions (fixed $T$, fixed endpoints).**
>
> $$\dot x = H_\lambda^\top = f, \qquad \dot \lambda = -H_x^\top, \qquad H_u = 0, \qquad x(0) = x_0, \quad x(T) = x_T .$$

There are no conditions on $\lambda(0)$ or $\lambda(T)$: with both ends of $x$
fixed, all eight boundary conditions are on the state. That is what makes this
a **two-point boundary value problem** (TPBVP): four conditions at $t = 0$,
four at $t = T$, on eight first-order ODEs.

In [ ]:
lam = sp.Matrix(sp.symbols("lam_q_1 lam_q_2 lam_qd_1 lam_qd_2", real=True))
H = L + (lam.T * f)[0]
lam_dot = -sp.Matrix([H]).jacobian(x).T
lam_dot[0], lam_dot[2]

The first and third costate equations are already informative:
$\dot\lambda_{q_1} = 0$, because $q_1$ does not appear in $f$. The track
looks the same everywhere, so the cart position is a **cyclic** coordinate and
its costate is constant. The other two are long. We let SymPy keep them.

### The control law

$H$ is quadratic in $u$ with $H_{uu} = 2 > 0$, so $H_u = 0$ has one solution
and it is a minimum.

In [ ]:
Hu = sp.diff(H, u)
u_star = sp.solve(Hu, u)[0]
sp.diff(H, u, 2), sp.simplify(u_star)

Because $f$ is affine in $u$, $H_u = 2u + \lambda^\top f_1$, so
$u^* = -\tfrac12 \lambda^\top f_1$. The control is a linear function of the
costate, through the input direction $f_1(x)$.

In [ ]:
sp.simplify(u_star + (lam.T * f1)[0] / 2)

## 3. The boundary value problem, and why it is hard

Substituting $u^*$ gives a closed system for $z = (x, \lambda)$:

$$
\dot z = F(z) = \begin{pmatrix} f(x, u^*(x, \lambda)) \\ -H_x(x, u^*(x, \lambda), \lambda)^\top \end{pmatrix},
\qquad x(0) = x_0, \quad x(T) = x_T .
$$

**Shooting** turns this into root-finding. Guess the missing half of
$z(0)$, the costate $\lambda_0$. Integrate $F$ from $(x_0, \lambda_0)$ to
$t = T$. Compare the state that arrives with the target:

$$
R(\lambda_0) = x(T; x_0, \lambda_0) - x_T = 0 .
$$

Four equations in four unknowns. Newton's method needs the Jacobian
$\partial R / \partial \lambda_0$, the sensitivity of the final state to the
initial costate. The costate is not a physical quantity, so there is little
intuition for guessing it. And because the costate equations run the
linearization *backwards* ($\dot\lambda = -A^\top\lambda$ near a
trajectory), errors in $\lambda_0$ grow fast. Both problems are classical
(AA203 lecture 5, "indirect methods: practical aspects"). Lecture 2 deals with
the guessing. The rest of this lecture is about the Jacobian.

In [ ]:
Hstar = H.subs(u, u_star)
z = sp.Matrix([*x, *lam])
F = sp.Matrix([*f.subs(u, u_star), *lam_dot.subs(u, u_star)])
F.shape

## 4. The state transition matrix

Perturb the initial condition, $z(0) \to z(0) + \delta z_0$. To first order the
perturbation obeys the **variational equation**, the linearization of $F$
along the trajectory:

$$
\dot{\delta z} = A(t)\,\delta z, \qquad A(t) = \frac{\partial F}{\partial z}\Big|_{z(t)} .
$$

It is linear, so $\delta z(t) = \Phi(t)\,\delta z_0$ for a matrix $\Phi(t)$,
the **state transition matrix** (STM). It solves the matrix ODE

$$
\dot \Phi = A(t)\,\Phi, \qquad \Phi(0) = I .
$$

Integrate it alongside $z$, $8 + 64$ equations in all, and the Newton
Jacobian is a block of $\Phi(T)$:
$\partial R / \partial \lambda_0 = \Phi_{x\lambda}(T)$, rows for $x$ and
columns for $\lambda$.

$A$ is the Jacobian of $F$, which SymPy differentiates exactly. That beats
finite differences, which need a step size and cost one extra integration
per unknown.

In [ ]:
A = F.jacobian(z)
sp.Matrix(8, 8, lambda i, j: 0 if A[i, j] == 0 else sp.Symbol("*"))

The structure is not an accident. With the control eliminated, the
canonical equations are $\dot x = H^*_\lambda$, $\dot\lambda = -H^*_x$ for
$H^*(x, \lambda) = H(x, u^*, \lambda)$, so $F = J \nabla H^*$ with
$J = \begin{pmatrix} 0 & I \\ -I & 0 \end{pmatrix}$. Then
$A = J\, \nabla^2 H^*$: the flow is Hamiltonian and $\Phi$ is symplectic.
The cell below checks $A = J \nabla^2 H^*$ at a random point (symbolic
simplification of the full $8 \times 8$ difference is slow).

In [ ]:
J = sp.Matrix(
    sp.BlockMatrix([[sp.zeros(4), sp.eye(4)], [-sp.eye(4), sp.zeros(4)]])
)
hess = sp.hessian(Hstar, z)
rng = np.random.default_rng(1)
point = dict(zip([*z, m1, m2, l, g], rng.uniform(0.2, 1.5, 12)))
diff = (A - J * hess).subs(point).evalf()
float(max(abs(v) for v in diff))

**Parameter sensitivities** work the same way. If $p$ collects the
parameters ($m_1, \dots, T$), $\Psi = \partial z(t) / \partial p$ obeys
$\dot \Psi = A\,\Psi + \partial F / \partial p$, $\Psi(0) = 0$. That is what
continuation in a parameter needs: how the solution moves when $p$ moves.

## 5. Bounded controls: the minimum principle

Real motors saturate: $|u| \le u_{\max}$. Then $\delta u$ is no longer
arbitrary at the bound (you cannot push past it), and $H_u = 0$ is the wrong
condition. Pontryagin's **minimum principle** replaces it with

$$
u^*(t) = \arg\min_{|v| \le u_{\max}} H(x^*(t), v, \lambda(t)) ,
$$

while the state and costate equations stay as they were (MIT 16.323 lecture
9; AA203 lecture 5; Liberzon chapter 4).

**Quadratic $H$ (our $L = u^2$).** The minimizer over an interval of a
convex parabola is its vertex, clipped to the interval:
$u^* = \operatorname{clip}(-\tfrac12\lambda^\top f_1, -u_{\max}, u_{\max})$.
The law is continuous but has kinks where it meets a bound. Lecture 3 solves
this case.

**Affine $H$ (minimum time, $L = 1$).** Now
$H = 1 + \lambda^\top f_0 + (\lambda^\top f_1)\, u$ is linear in $u$, and a
linear function on an interval is minimized at an end. Which end depends on
the sign of the coefficient of $u$, the **switching function**

$$
S(x, \lambda) = \frac{\partial H}{\partial u} = \lambda^\top f_1(x),
\qquad
u^* = \begin{cases} +u_{\max} & S < 0 \\ -u_{\max} & S > 0 . \end{cases}
$$

The control is **bang-bang**: it jumps between the bounds wherever $S$ crosses
zero. If $S$ stays at zero over an interval, the principle says nothing
there. That is a **singular arc**, and finding the control on it takes more
derivatives of $S$ (Liberzon section 4.4.3). Lecture 4 meets this case.

With a free final time, the transversality condition adds $H(T) = 0$. Our $H$
does not depend on $t$ explicitly, so it is constant along the extremal and
therefore $H \equiv 0$.

In [ ]:
S = sp.simplify((lam.T * f1)[0])
S

## 6. Switching and the jump in the STM

At a switch, $F$ is discontinuous: on one side the trajectory follows
$F^- = F(z, u = -u_{\max})$, on the other $F^+$. The trajectory itself stays
continuous, but $\Phi$ jumps. Here is the argument.

Take a nominal trajectory that reaches $S = 0$ at time $t_s$, and a perturbed
one, $z + \delta z$. The perturbed trajectory switches at $t_s + \delta t_s$,
where $S$ is still zero. To first order,

$$
\nabla S \cdot \delta z^-(t_s) + \dot S\, \delta t_s = 0
\quad \Rightarrow \quad
\delta t_s = -\frac{\nabla S \cdot \delta z^-}{\dot S},
\qquad \dot S = \frac{\partial S}{\partial t} + \nabla S \cdot F^- .
$$

Between $t_s$ and $t_s + \delta t_s$ one trajectory runs on $F^-$ and the
other on $F^+$, so just after the switch

$$
\delta z^+ = \delta z^- + (F^- - F^+)\,\delta t_s
= \Big( I + \frac{(F^+ - F^-)\, \nabla S^\top}{\dot S} \Big)\, \delta z^- .
$$

> **The saltation matrix.** At a switch,
>
> $$\Phi^+ = \Big(I + \frac{(F^+ - F^-)\,\nabla S^\top}{\dot S}\Big)\,\Phi^- .$$

This needs $\dot S \ne 0$, a transversal crossing. At a tangency the switch
time is not differentiable in $\delta z$, and Newton's method has trouble
there. The generated kernels need $\nabla S$ and $\dot S$ at every switch,
which is why `switching_grad` exists.

In [ ]:
grad_S = sp.Matrix([S]).jacobian(z)
[i for i in range(8) if grad_S[i] != 0]

$S$ depends on $q_2$ (through $f_1$) and on two of the costates, not on the
cart position or the velocities. Only those three columns of the jump are
nonzero.

## 7. What the package derives

`auto_optimal_control` does all of the above from the problem statement.
Below we pose the same three problems: minimum effort, minimum effort with a
force limit, and minimum time with a force limit. We check that `derive`
agrees with the hand derivation.

In [ ]:
from auto_optimal_control.pontryagin import derive
from auto_optimal_control.problem import Box, OptimalControlProblem

umax = sp.Symbol("u_max", positive=True)


def cart_pole(kind):
    common = dict(
        time=t,
        states=[q1, q2, w1, w2],
        controls=[u],
        dynamics=list(f),
        initial_state=[0, 0, 0, 0],
        final_state={q1: d, q2: sp.pi, w1: 0, w2: 0},
    )
    if kind == "effort":
        return OptimalControlProblem(
            name="cart_pole_effort",
            params=[m1, m2, l, g, d, T],
            running_cost=u**2,
            final_time=T,
            **common,
        )
    if kind == "bounded":
        return OptimalControlProblem(
            name="cart_pole_bounded",
            params=[m1, m2, l, g, umax, d, T],
            running_cost=u**2,
            final_time=T,
            control_sets=[Box(u, -umax, umax)],
            **common,
        )
    return OptimalControlProblem(
        name="cart_pole_min_time",
        params=[m1, m2, l, g, umax, d],
        running_cost=1,
        control_sets=[Box(u, -umax, umax)],
        **common,
    )


effort = derive(cart_pole("effort"))
effort

In [ ]:
rename = dict(zip(effort.costates, lam))
checks = {
    "hamiltonian": sp.simplify(effort.hamiltonian_full.subs(rename) - H),
    "control law": sp.simplify(effort.control_law[u].subs(rename) - u_star),
    "costate equations": sp.simplify(
        sp.Matrix(effort.costate_dynamics).subs(rename)
        - lam_dot.subs(u, u_star)
    ),
}
checks

In [ ]:
bounded = derive(cart_pole("bounded"))
bounded.control_law[u]

In [ ]:
min_time = derive(cart_pole("min_time"))
sw = min_time.switching_functions[0]
sw, sp.simplify(sw.expr.subs(dict(zip(min_time.costates, lam))) - S)

All three agree with the hand derivation: the bounded law is the clip of the
free one, and the minimum-time switching function is exactly
$S = \lambda^\top f_1$, with $u = +u_{\max}$ where $S < 0$.

The package poses the boundary value problem in a general form that also
covers partly free endpoints. The initial state is an unknown pinned by four
residuals. The terminal conditions $\psi(x(t_f)) = 0$ are adjoined with
multipliers $\nu$, which gives the transversality condition
$\lambda(t_f) = \psi_x^\top \nu$. The minimum-time problem adds one more
unknown, $t_f$, and one more condition, $H(t_f) = 0$. That makes 13 unknowns
and 13 residuals.

In [ ]:
min_time.boundary.unknowns, len(min_time.boundary.residuals)

### The generated C

`build_kernels` turns the system into C (via SymPy's code printer, with common
subexpressions eliminated), compiles it, and wraps it for the integrator. Here
is the start of the right-hand side $F$ for the minimum-time problem, and its
switching function.

In [ ]:
from auto_optimal_control.codegen import generate_c

prefix, header, source, macros = generate_c(min_time)
start = source.index(f"int {prefix}switching(")
print(source[start : source.index("\n}", start) + 2])

### The jump, checked numerically

Finally, compile the minimum-time kernels and check the saltation-matrix
argument. From an arbitrary initial costate, integrate for one second with the
STM. The bang-bang law switches along the way. Compare $\Phi(1)$ with central
finite differences of $z(1)$ in $z(0)$. If the jump were left out, the two
would disagree to first order after the first switch.

In [ ]:
from auto_optimal_control.build import build_kernels

k = build_kernels(min_time)
p = [1.0, 0.3, 0.5, 9.81, 20.0, 1.0]  # m1, m2, l, g, u_max, d
z0 = np.array([0, 0, 0, 0, 1.0, -2.0, 0.5, 0.3])

r = k.propagate(0.0, z0, 1.0, p, stm=True)
print("switches at t =", np.round(r["switch_times"], 4))

h = 1e-6


def z_at_1(z_start):
    return k.propagate(0.0, z_start, 1.0, p)["z"]


fd = np.column_stack(
    [(z_at_1(z0 + h * e) - z_at_1(z0 - h * e)) / (2 * h) for e in np.eye(8)]
)
print("max |Phi - finite differences| =", np.abs(r["stm"] - fd).max())

## Summary

- Adjoining the dynamics with a costate and setting the first variation to
  zero gives the canonical equations $\dot x = f$, $\dot\lambda = -H_x^\top$,
  the control condition, and boundary conditions split between the two ends.
- With bounds on the control, $H_u = 0$ becomes "minimize $H$ over the
  admissible set". A quadratic $H$ gives a clipped law. An affine $H$ gives a
  bang-bang law, decided by the sign of the switching function
  $S = \lambda^\top f_1$.
- Shooting needs $\partial z(T) / \partial z(0)$. The STM gives it exactly by
  integrating the variational equation $\dot\Phi = A\Phi$, with
  $A = \partial F / \partial z$ from symbolic differentiation, and it jumps
  by the saltation matrix at every switch.

## Exercises

1. $\lambda_{q_1}$ is constant because $f$ does not depend on $q_1$. Add
   viscous friction $-b\,\dot q_1$ to the cart's equation. Is $\lambda_{q_1}$
   still constant? Which costate equation changes, and how?
2. Repeat the variation of section 2 with $x(T)$ free and a terminal cost
   $\phi(x(T))$. Show that $\lambda(T) = \phi_x(x(T))^\top$.
3. In section 6, change $F^+$ and $F^-$ around and confirm the saltation
   matrix still maps $\delta z^-$ to $\delta z^+$ correctly. Then remove the
   jump by hand (integrate $\Phi$ across the switch with no correction) and
   measure the error against finite differences.

## Further reading

- D. E. Kirk, *Optimal Control Theory: An Introduction*, chapter 5.
- MIT 16.323, lecture notes 6 and 9 (J. How, 2008).
- Stanford AA203, lectures 4 and 5 (2026).
- D. Liberzon, *Calculus of Variations and Optimal Control Theory*,
  chapter 4.
- A. E. Bryson and Y.-C. Ho, *Applied Optimal Control*, chapter 2.
- M. Kelly (2017), SIAM Review 59(4), section 6, for the cart-pole problem.